# 4. Laplace approximation around a trained classifier

Train a small classifier, estimate curvature, and sample an explicit ensemble
using the current `LaplaceApproximation` API. See [setup instructions](README.md).
This CPU example uses synthetic data; no MNIST download or GPU is needed.

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from bensemble.core.ensemble import Ensemble
from bensemble.methods import LaplaceApproximation
from bensemble.metrics import negative_log_likelihood
from bensemble.uncertainty import decompose_classification_uncertainty

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)  # Small CPU workloads avoid thread-pool overhead.
DEVICE = torch.device("cpu")
EPOCHS = 250
NUM_MEMBERS = 20

## 1. Prepare data

Training data determine preprocessing, model weights, and curvature. We choose
the sampling temperature by validation NLL and evaluate once on the test split.

In [ ]:
# Generate two noisy crescents using only PyTorch.
generator = torch.Generator().manual_seed(SEED)
n = 900
labels = torch.arange(n) % 2
angle = torch.rand(n, generator=generator) * torch.pi
features = torch.stack((angle.cos(), angle.sin()), dim=1)
features[labels == 1, 0] = 1 - features[labels == 1, 0]
features[labels == 1, 1] = 0.5 - features[labels == 1, 1]
features += 0.25 * torch.randn(n, 2, generator=generator)

indices = torch.randperm(n, generator=generator)
split_indices = {
    "train": indices[:540],
    "validation": indices[540:720],
    "test": indices[720:],
}
x_train, y_train = features[split_indices["train"]], labels[split_indices["train"]]
x_val, y_val = (
    features[split_indices["validation"]],
    labels[split_indices["validation"]],
)
x_test, y_test = features[split_indices["test"]], labels[split_indices["test"]]

# Fit preprocessing on training data only, then reuse it on the held-out sets.
train_mean = x_train.mean(dim=0)
train_std = x_train.std(dim=0).clamp_min(1e-6)
x_train = (x_train - train_mean) / train_std
x_val = (x_val - train_mean) / train_std
x_test = (x_test - train_mean) / train_std

plt.figure(figsize=(6, 4))
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, cmap="coolwarm", s=12)
plt.xlabel("Standardized x1")
plt.ylabel("Standardized x2")
plt.title("Training data")
plt.show()

## 2. Train the base model with PyTorch

Laplace is a post-training approximation. Its constructor and curvature computation
do not train the model. Here an L2 penalty gives a MAP-style base fit.

In [ ]:
model = nn.Sequential(nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 2)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
prior_precision = 1.0
model.train()
for _ in range(EPOCHS):
    optimizer.zero_grad()
    data_loss = nn.functional.cross_entropy(model(x_train), y_train, reduction="sum")
    prior_loss = (
        0.5 * prior_precision * sum(p.square().sum() for p in model.parameters())
    )
    loss = (data_loss + prior_loss) / len(x_train)
    loss.backward()
    optimizer.step()
model.eval()

## 3. Estimate curvature and choose a sampling temperature

Set `prior_precision` on construction, then call `compute_curvature` with training
data. `num_samples` limits how many training examples contribute to the curvature
estimate. The default scales that estimate to the full training set before adding
the prior precision.

Temperature scales the weight perturbations. Compare three temperatures using the
same random draws and keep the ensemble with the lowest validation NLL.
The implementation samples linear-layer weights and keeps biases fixed.

In [ ]:
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=False)
laplace = LaplaceApproximation(
    model,
    likelihood="classification",
    prior_precision=prior_precision,
)
laplace.compute_curvature(train_loader, num_samples=len(x_train))

best_val_nll = float("inf")
with torch.no_grad(), torch.random.fork_rng(devices=[]):
    for temperature in (0.25, 0.5, 1.0):
        torch.manual_seed(SEED)
        candidate = Ensemble.from_posterior(
            laplace, n_members=NUM_MEMBERS, temperature=temperature
        ).eval()
        val_probs = candidate.predict_members(x_val).softmax(dim=-1).mean(dim=0)
        val_nll = negative_log_likelihood(val_probs, y_val)
        print(f"Temperature {temperature}: validation NLL={val_nll:.3f}")
        if val_nll < best_val_nll:
            best_val_nll = val_nll
            ensemble = candidate

## 4. Evaluate the predictive mixture

Average member **probabilities**, not their logits. Compare the fitted base model
and posterior mixture on the held-out test data. This small example demonstrates
the interface; a Laplace approximation is not guaranteed to improve accuracy or NLL.

In [ ]:
with torch.no_grad():
    base_probs = model(x_test).softmax(dim=-1)
    member_probs = ensemble.predict_members(x_test).softmax(dim=-1)
    test_probs = member_probs.mean(dim=0)

assert member_probs.shape == (NUM_MEMBERS, len(x_test), 2)
assert torch.isfinite(member_probs).all()
assert member_probs.var(dim=0).max() > 0
for name, probs in [("Base model", base_probs), ("Laplace ensemble", test_probs)]:
    accuracy = (probs.argmax(dim=-1) == y_test).float().mean().item()
    print(
        f"{name}: accuracy={accuracy:.3f}, NLL={negative_log_likelihood(probs, y_test):.3f}"
    )

total, aleatoric, epistemic = decompose_classification_uncertainty(member_probs)
print(f"Mean test predictive entropy: {total.mean():.3f}")
print(f"Mean test epistemic uncertainty: {epistemic.mean():.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
points = ax.scatter(x_test[:, 0], x_test[:, 1], c=epistemic, cmap="viridis", s=25)
fig.colorbar(points, ax=ax, label="Epistemic uncertainty (mutual information)")
ax.set(
    xlabel="Standardized x1",
    ylabel="Standardized x2",
    title="Laplace: test uncertainty",
)
plt.show()

Continue with [neural ensemble search](search/nes_rs.ipynb), or return to [the examples guide](README.md).